# Deploy Agents to AgentCore

Deploy all 5 agents to Amazon Bedrock AgentCore Runtime — serverless, VPC-connected, production-ready.

| Step | What Happens |
|---|---|
| 1 | Install dependencies |
| 2 | Verify environment |
| 3 | Deploy all agents |
| 4 | Test with `agentcore invoke` |

---
## Step 1: Install Dependencies

In [ ]:
!pip install bedrock-agentcore strands-agents[otel] bedrock-agentcore-starter-toolkit pymssql boto3 -q
!pip install 'urllib3<2' 'chardet<6' -q
print("✅ Dependencies installed")

---
## Step 2: Verify Environment Variables

In [ ]:
import os

required_vars = ['AWS_REGION', 'DB_INSTANCE_ID', 'DB_SECRET_ID', 'SNS_TOPIC_NAME',
                 'AGENTCORE_ROLE_ARN', 'SECURITY_GROUP_ID', 'SUBNET1']

print("╔══════════════════════════════════════════════════╗")
print("║  Environment Variables                          ║")
print("╠══════════════════════════════════════════════════╣")
all_set = True
for var in required_vars:
    val = os.getenv(var, '')
    status = '✅' if val else '❌'
    if not val:
        all_set = False
    print(f"  {status} {var}: {val[:40] if val else 'NOT SET'}")
print("╚══════════════════════════════════════════════════╝")

if not all_set:
    print("\n⚠️  Some variables are missing. Run:")
    print("export DB_SECRET_ID=dbops-infra-sqlserver-secret DB_INSTANCE_ID=dbops-infra-sqlserver SNS_TOPIC_NAME=sqlserver-database-alerts")

---
## Step 3: Enable X-Ray Observability

Set up CloudWatch Logs as the trace destination for GenAI Observability:

In [ ]:
%%bash
aws logs put-resource-policy \
  --policy-name XRaySpansLogGroupPolicy \
  --policy-document '{"Version":"2012-10-17","Statement":[{"Effect":"Allow","Principal":{"Service":"xray.amazonaws.com"},"Action":["logs:PutLogEvents","logs:CreateLogStream"],"Resource":"*"}]}' \
  --region $AWS_REGION

aws xray update-trace-segment-destination --destination CloudWatchLogs --region $AWS_REGION
echo "✅ X-Ray observability configured"

---
## Step 4: Deploy All Agents

This deploys all 5 agents to AgentCore Runtime with:
- Shared memory (semantic + summarization strategies)
- VPC configuration for database access
- Environment variables for each agent
- Supervisor wired to sub-agent ARNs

⏱️ Takes approximately 5-10 minutes.

In [ ]:
%%bash
cd /workshop/labs/agentcore_agents
chmod +x deploy_all_agents.sh
./deploy_all_agents.sh

---
## Step 5: Test Your Agents

Invoke the Supervisor agent — it routes your request to the right sub-agent(s):

In [ ]:
%%bash
agentcore invoke --agent supervisor_agent '{"prompt": "Give me a quick database health summary"}'

In [ ]:
%%bash
# Try individual agents directly
agentcore invoke --agent database_health_agent '{"prompt": "What is the current CPU utilization?"}'

In [ ]:
%%bash
agentcore invoke --agent query_performance_agent '{"prompt": "Show me the top 3 slowest queries"}'

---
## Step 6: Check Agent Status

In [ ]:
%%bash
echo "Agent Status:"
echo "============="
for agent in database_health_agent query_performance_agent security_audit_agent data_lifecycle_agent supervisor_agent; do
    STATUS=$(agentcore status --agent $agent 2>/dev/null | grep -o '"status": "[^"]*"' | head -1 | cut -d'"' -f4)
    echo "  $agent: $STATUS"
done

---
## Key Takeaways

- Agents deploy as **serverless containers** in your VPC
- **Shared memory** enables cross-agent knowledge sharing
- **GenAI Observability** tracks token usage, latency, and traces
- **`agentcore invoke`** calls any agent on-demand

**Next:** Set up autonomous remediation with safety guardrails.